In [2]:
from transformers import pipeline
from datasets import load_dataset
from tqdm import tqdm
import numpy as np


In [3]:
dataset = load_dataset("squad")


d:\qa-transformers\qa_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\datasets--squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 10570/10570 [00:00<00:00, 210495.85 examples/s]


In [4]:
qa_pipeline = pipeline("question-answering", model="bert-large-uncased-whole-word-masking-finetuned-squad")


Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [ ]:
def get_predictions(dataset, num_samples=100):
    predictions = []
    references = []
    for item in tqdm(dataset['validation']):
        context = item['context']
        question = item['question']
        expected = item['answers']['text'][0]

        result = qa_pipeline(question=question, context=context)
        predictions.append(result['answer'])
        references.append(expected)
    return predictions, references

preds, refs = get_predictions(dataset)


  7%|▋         | 736/10570 [20:10<4:27:39,  1.63s/it] 

In [ ]:
def exact_match(pred, truth):
    return int(pred.strip().lower() == truth.strip().lower())

def f1_score(pred, truth):
    pred_tokens = pred.lower().split()
    truth_tokens = truth.lower().split()
    common = set(pred_tokens) & set(truth_tokens)
    if not common:
        return 0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(truth_tokens)
    return 2 * (precision * recall) / (precision + recall)

em_scores = [exact_match(p, t) for p, t in zip(preds, refs)]
f1_scores = [f1_score(p, t) for p, t in zip(preds, refs)]

print(f"Exact Match (EM): {np.mean(em_scores) * 100:.2f}%")
print(f"F1 Score: {np.mean(f1_scores) * 100:.2f}%")


Exact Match (EM): 77.00%
F1 Score: 82.85%
